# Nivel 3: Retos avanzados y rendimiento

Ejercicios 3.1 a 3.4 (problemas más elaborados) y P.1 y P.2 (medición de rendimiento).

In [1]:
import math
import time
from pathlib import Path
from mapreduce_framework import mapreduce, print_results

BOOK = Path("data/book/The story of the universe.txt")
PUNCT = '.,!?;:"()[]{}'


def read_lines(path):
    """Lee un archivo y devuelve sus líneas no vacías."""
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


book_lines = read_lines(BOOK)
print(f"Libro: {len(book_lines)} líneas")

Libro: 11992 líneas


---
## Ejercicio 3.1: Top-N palabras

**Objetivo:** encontrar las N palabras más frecuentes. MapReduce cuenta las palabras y luego se ordena el resultado para quedarse con las N primeras.

In [2]:
def mapper(line):
    """Emite (palabra, 1) por cada palabra."""
    for word in line.lower().split():
        word = word.strip(PUNCT)
        if word:
            yield (word, 1)


def reducer(word, counts):
    """Suma las apariciones de una palabra."""
    return sum(counts)


def top_n_words(text, n=10):
    """Devuelve las n palabras más frecuentes como lista de (palabra, conteo)."""
    results = mapreduce(text, mapper, reducer)
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:n]


print("Top 10 palabras del libro:")
for word, count in top_n_words(book_lines, n=10):
    print(f"  {word:10} {count:6}")

print("\nDatos vacíos:", top_n_words([], n=10))

Top 10 palabras del libro:
  the         10873
  of           6288
  and          4078
  in           3115
  a            2681
  to           2384
  is           1807
  are          1420
  which        1366
  it           1094

Datos vacíos: []


---
## Ejercicio 3.2: Análisis de bigramas

**Objetivo:** encontrar los pares de palabras consecutivas más comunes.

- **mapper** → emite `("palabra1 palabra2", 1)` por cada par consecutivo.
- **reducer** → suma.

In [3]:
text = ["the quick brown fox", "the quick red dog"]


def mapper(line):
    """Emite (bigrama, 1) por cada par de palabras consecutivas."""
    words = [w.strip(PUNCT) for w in line.lower().split()]
    words = [w for w in words if w]
    for i in range(len(words) - 1):
        yield (f"{words[i]} {words[i + 1]}", 1)


def reducer(bigram, counts):
    """Suma las apariciones de un bigrama."""
    return sum(counts)


print_results(mapreduce(text, mapper, reducer), "Bigramas del ejemplo")
print_results(mapreduce(book_lines, mapper, reducer), "Top 10 bigramas del libro", limit=10)

print("Datos vacíos:", mapreduce([], mapper, reducer))


Bigramas del ejemplo
the quick: 2
quick brown: 1
brown fox: 1
quick red: 1
red dog: 1


Top 10 bigramas del libro
of the: 1809
in the: 919
to the: 591
and the: 351
it is: 348
of a: 335
on the: 313
from the: 283
by the: 267
with the: 187

Datos vacíos: {}


---
## Ejercicio 3.3: Análisis de sesiones web

**Objetivo:** para cada usuario, calcular el número de páginas visitadas y la lista de páginas únicas.

Como extra, el mapper emite `(hora, página)` en lugar de solo la página. Así el reducer puede reconstruir el **recorrido** del usuario en orden.

In [4]:
events = [
    {'user': 'U1', 'page': '/home', 'time': '10:00'},
    {'user': 'U1', 'page': '/products', 'time': '10:02'},
    {'user': 'U2', 'page': '/home', 'time': '10:01'},
    {'user': 'U1', 'page': '/cart', 'time': '10:05'},
    {'user': 'U2', 'page': '/about', 'time': '10:03'},
]


def mapper(event):
    """Emite (usuario, (hora, página))."""
    yield (event['user'], (event['time'], event['page']))


def reducer(user, visits):
    """Calcula páginas vistas, páginas únicas y el recorrido en orden."""
    visits = sorted(visits)                     # ordena por hora
    pages = [page for _, page in visits]
    return {
        'page_views': len(pages),
        'unique_pages': sorted(set(pages)),
        'path': " -> ".join(pages),
    }


for user, session in mapreduce(events, mapper, reducer).items():
    print(f"{user}: {session}")

print("Datos vacíos:", mapreduce([], mapper, reducer))

U1: {'page_views': 3, 'unique_pages': ['/cart', '/home', '/products'], 'path': '/home -> /products -> /cart'}
U2: {'page_views': 2, 'unique_pages': ['/about', '/home'], 'path': '/home -> /about'}
Datos vacíos: {}


---
## Ejercicio 3.4: Detector de anomalías

**Objetivo:** para cada sensor, calcular el promedio y marcar los valores que se alejan más de 2 desviaciones estándar.

- **mapper** → emite `(sensor, valor)`.
- **reducer** → promedio, desviación estándar y anomalías.

In [5]:
readings = [
    {'sensor': 'S1', 'value': 22.5},
    {'sensor': 'S1', 'value': 23.0},
    {'sensor': 'S1', 'value': 99.9},  # <- anomalía
    {'sensor': 'S2', 'value': 15.0},
    {'sensor': 'S2', 'value': 14.8},
]


def mapper(reading):
    """Emite (sensor, valor)."""
    yield (reading['sensor'], reading['value'])


def reducer(sensor, values):
    """Promedio, desviación estándar y valores a más de 2 desviaciones del promedio."""
    avg = sum(values) / len(values)
    variance = sum((v - avg) ** 2 for v in values) / len(values)
    std_dev = math.sqrt(variance)
    anomalies = [v for v in values if abs(v - avg) > 2 * std_dev]
    return {
        'avg': round(avg, 2),
        'std_dev': round(std_dev, 2),
        'anomalies': anomalies,
    }


for sensor, stats in mapreduce(readings, mapper, reducer).items():
    print(f"{sensor}: {stats}")

S1: {'avg': 48.47, 'std_dev': 36.37, 'anomalies': []}
S2: {'avg': 14.9, 'std_dev': 0.1, 'anomalies': []}


### ⚠️ Observación: con 2 desviaciones estándar no se detecta el 99.9

El método del enunciado **no encuentra la anomalía**. No es un error del código, es una limitación estadística: con solo `n` valores, ningún punto puede alejarse más de $\sqrt{n-1}$ desviaciones del promedio. Con 3 valores, el máximo posible es $\sqrt{2} \approx 1.41$, nunca 2.

Además, el propio 99.9 "infla" el promedio (48.47) y la desviación (36.37), y así se esconde a sí mismo.

**Alternativa robusta:** usar la **mediana** y la **MAD** (mediana de las desviaciones absolutas), que no se dejan arrastrar por un valor extremo. Se marca como anomalía lo que se aleje más de 3 MAD de la mediana.

In [6]:
def median(values):
    """Mediana de una lista de números."""
    s = sorted(values)
    mid = len(s) // 2
    return s[mid] if len(s) % 2 else (s[mid - 1] + s[mid]) / 2


def robust_reducer(sensor, values):
    """Detecta anomalías con mediana y MAD (más de 3 MAD de la mediana)."""
    med = median(values)
    mad = median([abs(v - med) for v in values])
    if mad == 0:                         # todos iguales: no hay anomalías
        return {'median': med, 'mad': 0, 'anomalies': []}
    anomalies = [v for v in values if abs(v - med) > 3 * mad]
    return {'median': med, 'mad': round(mad, 2), 'anomalies': anomalies}


for sensor, stats in mapreduce(readings, mapper, robust_reducer).items():
    print(f"{sensor}: {stats}")

print("Datos vacíos:", mapreduce([], mapper, robust_reducer))

S1: {'median': 23.0, 'mad': 0.5, 'anomalies': [99.9]}
S2: {'median': 14.9, 'mad': 0.1, 'anomalies': []}
Datos vacíos: {}


---
## Ejercicio P.1: Secuencial vs paralelo

**Objetivo:** comparar el tiempo de procesar el libro con el framework básico (secuencial) y con `parallel_mapreduce.py` (varios procesos).

El mapper y el reducer se importan desde `parallel_mapreduce.py` porque, en Windows, los procesos hijos solo pueden usar funciones definidas en un archivo `.py`, no en el notebook.

In [7]:
import contextlib
import io
from parallel_mapreduce import parallel_mapreduce, word_mapper, word_reducer

# Secuencial
start = time.time()
seq_results = mapreduce(book_lines, word_mapper, word_reducer)
seq_time = time.time() - start

# Paralelo (se silencian sus mensajes internos para ver solo el resumen)
start = time.time()
with contextlib.redirect_stdout(io.StringIO()):
    par_results = parallel_mapreduce(book_lines, word_mapper, word_reducer,
                                     num_mappers=4, num_reducers=2)
par_time = time.time() - start

print(f"Líneas procesadas: {len(book_lines)}")
print(f"Secuencial: {seq_time:.3f}s")
print(f"Paralelo:   {par_time:.3f}s")
print(f"Mismo resultado: {seq_results == par_results}")

Líneas procesadas: 11992
Secuencial: 0.069s
Paralelo:   0.657s
Mismo resultado: True


### Análisis

Con un archivo de este tamaño (~750 KB), el paralelo suele ser **más lento** que el secuencial. Crear procesos y enviarles los datos cuesta más tiempo que el trabajo en sí, que es muy poco.

El paralelismo compensa cuando el volumen de datos es grande (cientos de MB o GB). Por eso Hadoop y Spark se usan para Big Data y no para archivos pequeños.

---
## Ejercicio P.2: Efecto del tamaño de bloque

**Objetivo:** ver cómo el tamaño de bloque del HDFS simulado afecta la cantidad de bloques y el tiempo total.

En lugar de modificar `distributed_mapreduce.py` a mano, se crea un `SimulatedHDFS` con cada tamaño de bloque y se ejecuta el mismo trabajo.

In [8]:
import shutil
from simulated_hdfs import SimulatedHDFS
from distributed_mapreduce import distributed_mapreduce_from_hdfs

rows = []
for block_size in [4096, 16384, 65536]:
    storage = f"hdfs_storage_{block_size}"
    shutil.rmtree(storage, ignore_errors=True)       # empieza limpio

    start = time.time()
    with contextlib.redirect_stdout(io.StringIO()):  # silencia los mensajes internos
        hdfs = SimulatedHDFS(base_dir=storage, block_size=block_size, replication=3, num_nodes=6)
        hdfs.put(str(BOOK), "/data/book.txt")
        results = distributed_mapreduce_from_hdfs(hdfs, "/data/book.txt",
                                                  word_mapper, word_reducer,
                                                  num_mappers=4, num_reducers=2)
    elapsed = time.time() - start

    rows.append((block_size, len(hdfs.get_blocks("/data/book.txt")), elapsed,
                 results == seq_results))
    shutil.rmtree(storage, ignore_errors=True)       # borra el almacenamiento temporal

print(f"{'Bloque':>10} | {'Nº bloques':>10} | {'Tiempo':>8} | Resultado correcto")
print("-" * 55)
for block_size, n_blocks, elapsed, ok in rows:
    print(f"{block_size // 1024:>7} KB | {n_blocks:>10} | {elapsed:>7.3f}s | {ok}")

    Bloque | Nº bloques |   Tiempo | Resultado correcto
-------------------------------------------------------
      4 KB |        183 |   2.127s | True
     16 KB |         46 |   0.820s | True
     64 KB |         12 |   0.525s | True


### Análisis

- **Bloques pequeños (4 KB):** el archivo se divide en muchos bloques. Hay que escribir y leer muchos archivos, cada uno con 3 réplicas, y esa gestión hace más lento el proceso.
- **Bloques grandes (64 KB):** pocos bloques, menos gestión y menos tiempo.

Por eso HDFS usa bloques grandes (128 MB por defecto): procesar muchos bloques pequeños tiene un costo fijo por bloque que se acumula. El resultado del conteo es el mismo en todos los casos; solo cambia el rendimiento.